**Longest Repeated Substring (LRSS) via suffix arrays.**
A **repeated substring** is a contiguous sequence of characters that appears
at least twice in a string.
Finding the longest one has direct applications in bioinformatics - for example,
identifying repeated motifs in a DNA sequence.

The algorithm uses a **suffix array**: the sorted list of all suffixes of the
input string.
Because suffixes that share a long common prefix will sort adjacent to each
other, scanning consecutive pairs in the sorted list and measuring their
longest common prefix finds the longest repeated substring in $O(n \log n)$
time (dominated by the sort).

In [ ]:
"""seq_lrss.ipynb"""

# Cell 01 - Longest common prefix of two strings
# zip() naturally stops at the shorter string, so no length check is needed.


def longest_common_prefix(s1: str, s2: str) -> str:
    """Return the longest string that is a prefix of both s1 and s2."""
    prefix = []
    for c1, c2 in zip(s1, s2):
        if c1 == c2:
            prefix.append(c1)
        else:
            break
    return "".join(prefix)


print(longest_common_prefix("Hello", "He"))  # 'He'
print(longest_common_prefix("AACAAG", "AACAAG"))  # 'AACAAG'
print(longest_common_prefix("AAC", "AAT"))  # 'AA'
print(longest_common_prefix("ABC", "XYZ"))  # ''

**The suffix array algorithm.**
Given a string of length $n$, we form all $n$ suffixes
(substrings starting at each position), sort them lexicographically,
then compare each adjacent pair.
The longest common prefix found across all adjacent pairs is the answer.

For example, `"AACAAGTTTACAAGC"` has the suffixes (sorted):
```
AACAAGTTTACAAGC
AACAAGC            <- shares prefix ACAAG with the next suffix
ACAAGTTTACAAGC
ACAAGC             <- shares prefix ACAAG with the previous suffix
...
```
The longest repeated substring is `ACAAG`.

In [ ]:
# Cell 02 - Longest repeated substring via suffix array


def lrss(seq: str) -> str:
    """Return the longest substring of seq that appears at least twice."""
    # Step 1 - build all suffixes
    suffixes = [seq[i:] for i in range(len(seq))]

    # Step 2 - sort: suffixes sharing a long common prefix sort adjacent
    suffixes.sort()

    # Step 3 - scan adjacent pairs for the longest common prefix
    longest = ""
    for i in range(len(suffixes) - 1):
        candidate = longest_common_prefix(suffixes[i], suffixes[i + 1])
        if len(candidate) > len(longest):
            longest = candidate

    return longest


seq = "AACAAGTTTACAAGC"
result = lrss(seq)
print(f"Sequence : {seq}")
print(f"LRSS     : {result}")
print(f"Count    : {seq.count(result)} occurrences")

**Applying LRSS to a real DNA file.**
The file is read as raw bytes, decoded to a UTF-8 string, converted to
uppercase, then stripped of everything that is not an uppercase letter
using a regular expression.
This leaves a clean string of DNA bases (A, C, G, T) ready for analysis.

The compiled regex `_NON_LETTER` is defined once at module level so it
is not recompiled on every function call.

In [ ]:
# Cell 03 - Read a DNA sequence file and find its longest repeated substring

import re
from pathlib import Path

# Compile once at module level - matches any character that is not A-Z
_NON_LETTER = re.compile("[^A-Z]")


def analyze_file(file_name: str) -> None:
    """Read a DNA sequence file and print its longest repeated substring."""
    seq = Path(file_name).read_bytes().decode().upper()
    seq = _NON_LETTER.sub("", seq)

    longest = lrss(seq)
    print(f"File              : {file_name}")
    print(f"Sequence length   : {len(seq):,} bases")
    print(f"Longest repeated  : {longest}")
    print(f"Count             : {seq.count(longest)} occurrences")


analyze_file("fruitfly.txt")